# Stage 2 - judge (Kaggle **GPU T4 x2 required**, Internet ON)
Set the accelerator BEFORE running. Pilot = 8 files x 20 = 160 judgments; expect about 5-15 min on GPU (guess). The script aborts if the first judgments take over 60 s each. Exports only verdict files (no responses).

In [1]:
import os, glob, subprocess, torch
assert torch.cuda.is_available(), "NO GPU. Settings -> Accelerator -> GPU T4 x2, then restart. (The last attempt ran on CPU and timed out after 12 h.)"
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
GEN_ZIP = sorted(glob.glob("/kaggle/input/**/gen_*.zip", recursive=True))[0]; print(GEN_ZIP)
JUDGE_HF = "Qwen/Qwen2.5-7B-Instruct"; JUDGE_ID = "qwen25-7b"      # fp16 split over 2 GPUs; if memory errors: "Qwen/Qwen2.5-3B-Instruct" and JUDGE_ID = "qwen25-3b"
!git clone -q https://github.com/sarrazer24/reins-mini /kaggle/working/reins-mini
%cd /kaggle/working/reins-mini
!mkdir -p runs && unzip -qo {GEN_ZIP} -d runs
!pip install -q -U transformers accelerate
!git clone -q --depth 1 https://github.com/Geralt1020/REINS /tmp/REINS

GPUs: 2 ['Tesla T4', 'Tesla T4']
/kaggle/input/notebooks/sarrazer/kaggle-10-generate/reins-mini/gen_pilot_seed0.zip
/kaggle/working/reins-mini


In [2]:
for d in sorted(glob.glob("runs/*/seed*")):
    print("judging", d, flush=True)
    subprocess.run(f"python -u scripts/11_run_judge.py --run-dir {d} --guise-dir /tmp/REINS/guise --judge-id {JUDGE_ID} --hf-model {JUDGE_HF}", shell=True, check=True)

judging runs/pilot/seed0


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 339/339 [00:31<00:00, 10.61it/s]


  harm_only.jsonl 1/20  6.3s per judgment
  harm_only.jsonl 2/20  4.3s per judgment
  harm_only.jsonl 3/20  3.7s per judgment
  harm_only.jsonl 5/20  3.1s per judgment
  harm_only.jsonl 10/20  2.7s per judgment
  harm_only.jsonl 15/20  2.6s per judgment
  harm_only.jsonl 20/20  2.5s per judgment
harm_only.jsonl judged 20
  original.jsonl 5/20  2.5s per judgment
  original.jsonl 10/20  2.5s per judgment
  original.jsonl 15/20  2.4s per judgment
  original.jsonl 20/20  2.4s per judgment
original.jsonl judged 20
  original_safe.jsonl 5/20  2.4s per judgment
  original_safe.jsonl 10/20  2.4s per judgment
  original_safe.jsonl 15/20  2.4s per judgment
  original_safe.jsonl 20/20  2.4s per judgment
original_safe.jsonl judged 20
  random_active.jsonl 5/20  2.4s per judgment
  random_active.jsonl 10/20  2.4s per judgment
  random_active.jsonl 15/20  2.4s per judgment
  random_active.jsonl 20/20  2.4s per judgment
random_active.jsonl judged 20
  random_any.jsonl 5/20  2.4s per judgment
  random

In [3]:
!find runs -name "*.judged_*.jsonl" -o -name "gate_scores_*.csv" | zip -q judged_{JUDGE_ID}.zip -@
!ls -la judged_*.zip
!rm -rf /tmp/REINS

-rw-r--r-- 1 root root 5841 Nov 16 09:27 judged_qwen25-7b.zip
